In [1]:
import json
import pandas as pd
import numpy as np
from datetime import datetime, date
from pathlib import Path

In [2]:
DATA_PATH = Path("data/candidates.jsonl")  
REFERENCE_DATE = date(2026, 6, 14)
print("Loading 100K candidates...")
with open(DATA_PATH, "rt", encoding="utf-8") as f:
    candidates = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(candidates):,} candidates")

# Build a lookup dict for fast access by ID
cand_lookup = {c["candidate_id"]: c for c in candidates}
print("Lookup dict built.")

Loading 100K candidates...
Loaded 100,000 candidates
Lookup dict built.


In [3]:
def days_since(date_str: str) -> int:
    try:
        d = datetime.strptime(date_str, "%Y-%m-%d").date()
        return (REFERENCE_DATE - d).days
    except:
        return 9999

In [4]:
JD = {
    "required_skills": {
        "sentence-transformers", "embeddings", "dense retrieval",
        "semantic search", "RAG", "retrieval", "text embeddings",
        "BGE", "E5", "OpenAI embeddings", "embedding models",
        "Pinecone", "Weaviate", "Qdrant", "Milvus", "FAISS",
        "Elasticsearch", "OpenSearch", "vector database", "vector search",
        "hybrid search",
        "ranking", "information retrieval", "BM25", "learning to rank",
        "LTR", "NDCG", "MRR", "reranking", "re-ranking",
        "recommendation systems", "recommender systems",
        "LLM", "fine-tuning", "LoRA", "QLoRA", "PEFT", "instruction tuning",
        "NLP", "transformers", "BERT", "language models",
        "Python",
        "A/B testing", "offline evaluation", "eval framework",
    },
    "bonus_skills": {
        "XGBoost", "LightGBM", "learning to rank", "distributed systems",
        "MLflow", "Weights & Biases", "experiment tracking",
        "FastAPI", "Triton", "ONNX", "model serving", "inference optimization",
        "Kubernetes", "Docker", "MLOps",
        "open source", "GitHub", "research papers",
        "HR tech", "recruiting tech", "marketplace",
        "Spark", "distributed training", "Ray",
    },
    "disqualifier_titles": {
        "accountant", "marketing manager", "operations manager",
        "civil engineer", "mechanical engineer", "customer support",
        "sales manager", "hr manager", "finance manager",
        "business analyst", "project manager", "product manager",
        "graphic designer", "ui designer", "ux designer",
        "content writer", "seo specialist", "social media",
        "recruiter", "talent acquisition",
    },
    "consulting_firms": {
        "TCS", "Infosys", "Wipro", "Accenture", "Cognizant",
        "Capgemini", "HCL", "Mphasis", "Tech Mahindra", "Hexaware",
        "Mindtree", "L&T Infotech", "LTIMindtree", "Persistent Systems",
        "NIIT Technologies", "Mastech",
    },
    "good_industries": {
        "Software", "Fintech", "E-commerce", "SaaS", "AI", "ML",
        "Technology", "Internet", "Product", "Transportation",
        "Food Delivery",
    },
    "bad_industries": {
        "Manufacturing", "Paper Products", "Construction",
        "Accounting", "Legal", "Healthcare Administration",
    },
    "good_titles": {
        # Specific ML/AI titles — get ×1.0 title multiplier
        "recommendation systems engineer",
        "machine learning engineer", "ml engineer",
        "ai engineer", "applied scientist", "senior applied scientist",
        "research scientist", "nlp engineer",
        "search engineer", "ranking engineer",
        "retrieval engineer", "data scientist",
        "computer scientist", "lead ai engineer",
        "senior nlp engineer", "senior ai engineer",
        "senior machine learning engineer", "staff machine learning engineer",
        "applied ml engineer", "ai research engineer",
        "junior ml engineer",
    },
    "neutral_eng_titles": {
        # Generic eng titles — get ×0.5-0.8 multiplier depending on ML history
        "software engineer", "senior engineer", "staff engineer",
        "principal engineer", "cloud engineer", "devops engineer",
        "platform engineer", "full stack", "frontend engineer",
        "backend engineer", "data engineer",
    },
    "preferred_locations": {
        "Pune", "Noida", "Bengaluru", "Bangalore", "Hyderabad",
        "Mumbai", "Delhi", "Gurgaon", "Gurugram", "Chennai",
    },
    "preferred_countries": {"India"},
    "yoe_ideal_min": 5,
    "yoe_ideal_max": 9,
    "yoe_hard_min":  3,
    "yoe_hard_max":  14,
}

print("JD config loaded.")
print(f"  Required skills:  {len(JD['required_skills'])}")
print(f"  Bonus skills:     {len(JD['bonus_skills'])}")
print(f"  Disq titles:      {len(JD['disqualifier_titles'])}")
print(f"  Good titles:      {len(JD['good_titles'])}")
print(f"  Neutral eng titles: {len(JD['neutral_eng_titles'])}")

JD config loaded.
  Required skills:  46
  Bonus skills:     24
  Disq titles:      20
  Good titles:      21
  Neutral eng titles: 11


In [5]:
# Represent the JD as a rich text block for embedding
# More text = better signal — include context, not just skill names
JD_TEXT = """
Senior AI Engineer at an AI-native talent intelligence platform.

Required experience: production embeddings-based retrieval systems using
sentence-transformers, BGE, E5, OpenAI embeddings or similar. Vector databases
and hybrid search infrastructure — Pinecone, Weaviate, Qdrant, Milvus, FAISS,
Elasticsearch, OpenSearch. Strong Python. Evaluation frameworks for ranking
systems — NDCG, MRR, MAP, A/B testing, offline to online correlation.

Nice to have: LLM fine-tuning with LoRA, QLoRA, PEFT. Learning to rank with
XGBoost or neural models. Experience with HR tech or marketplace products.
Distributed systems or large-scale inference optimization. Open source
contributions in AI/ML.

Ideal candidate: 6-8 years total experience, 4-5 years in applied ML at product
companies. Has shipped at least one end-to-end ranking, search, or recommendation
system to real users at meaningful scale. Strong opinions about retrieval,
evaluation, and LLM integration backed by systems they actually built.
Located in or willing to relocate to Pune or Noida India.

Not a fit: pure research background with no production deployment. Only recent
LangChain API-wrapping experience. Entire career at TCS Infosys Wipro Accenture
Cognizant Capgemini. Primary expertise in computer vision speech or robotics
without NLP or information retrieval. Title chaser switching companies every
18 months.
"""

print("JD text ready.")
print(f"Length: {len(JD_TEXT.split())} words")

JD text ready.
Length: 189 words


In [6]:
import os
JD_FILE = "data/job_description.md"
if os.path.exists(JD_FILE):
    with open(JD_FILE) as f:
        JD_TEXT = f.read()
    print("JD loaded from file.")
else:
    # JD_TEXT already defined above
    print("Using hardcoded JD_TEXT.")

Using hardcoded JD_TEXT.


In [7]:
def score_career(candidate: dict, jd: dict) -> tuple[float, dict]:
    history  = candidate["career_history"]
    profile  = candidate["profile"]
    yoe      = profile["years_of_experience"]

    consulting_firms = {f.lower() for f in jd["consulting_firms"]}
    good_titles      = {t.lower() for t in jd["good_titles"]}
    disq_titles      = {t.lower() for t in jd["disqualifier_titles"]}

    total_months      = sum(h.get("duration_months", 0) for h in history)
    consulting_months = 0
    relevant_months   = 0
    short_stints      = 0

    # Also check past titles for ML relevance
    past_ml_title = False

    for h in history:
        company  = h.get("company", "").lower()
        title    = h.get("title", "").lower()
        dur      = h.get("duration_months", 0)

        is_consulting = any(firm in company for firm in consulting_firms)
        is_ml_title   = any(gt in title for gt in good_titles)

        if is_consulting:
            consulting_months += dur
        if is_ml_title:
            relevant_months += dur
            past_ml_title = True
        if dur < 12 and not h.get("is_current"):
            short_stints += 1

    consulting_ratio = consulting_months / max(total_months, 1)
    relevant_ratio   = relevant_months / max(total_months, 1)

    # --- Current title scoring (stricter) ---
    current_title = profile["current_title"].lower()
    is_disq       = any(dt in current_title for dt in disq_titles)
    is_good       = any(gt in current_title for gt in good_titles)

    if is_disq and not past_ml_title:
        # Clearly wrong domain AND no prior ML history
        title_score = 0.0
    elif is_disq and past_ml_title:
        # Wrong current title but has ML history — transitioning out?
        title_score = 0.2
    elif is_good:
        title_score = 1.0
    elif past_ml_title:
        # Generic title (e.g. "Software Engineer") but has ML in past
        title_score = 0.6 + 0.3 * relevant_ratio
    else:
        # Generic non-ML title, no ML history
        title_score = 0.15

    # --- YoE score ---
    if yoe < jd["yoe_hard_min"]:
        yoe_score = 0.1
    elif yoe < jd["yoe_ideal_min"]:
        yoe_score = 0.5 + 0.5 * (yoe - jd["yoe_hard_min"]) / (jd["yoe_ideal_min"] - jd["yoe_hard_min"])
    elif yoe <= jd["yoe_ideal_max"]:
        yoe_score = 1.0
    elif yoe <= jd["yoe_hard_max"]:
        yoe_score = 1.0 - 0.3 * (yoe - jd["yoe_ideal_max"]) / (jd["yoe_hard_max"] - jd["yoe_ideal_max"])
    else:
        yoe_score = 0.5

    # --- Consulting ratio score ---
    if consulting_ratio > 0.9:
        consulting_score = 0.05
    elif consulting_ratio > 0.6:
        consulting_score = 0.3
    elif consulting_ratio > 0.3:
        consulting_score = 0.7
    else:
        consulting_score = 1.0

    # --- Job hopping penalty ---
    hop_penalty = min(short_stints * 0.1, 0.3)

    career_score = (
        0.40 * title_score +
        0.28 * consulting_score +
        0.25 * yoe_score +
        0.07 * (1.0 - hop_penalty)
    )

    return career_score, {
        "yoe_score":        round(yoe_score, 3),
        "title_score":      round(title_score, 3),
        "consulting_ratio": round(consulting_ratio, 3),
        "consulting_score": round(consulting_score, 3),
        "relevant_ratio":   round(relevant_ratio, 3),
        "past_ml_title":    past_ml_title,
        "short_stints":     short_stints,
        "final":            round(career_score, 3),
    }

print("Career scorer redefined.")

Career scorer redefined.


In [8]:
def score_behavioral(candidate: dict) -> tuple[float, dict]:
    """
    Returns (multiplier, detail_dict).
    
    Structure per the spec:
      Base availability multiplier (Group A): 0.3 – 1.0
      Engagement bonus/penalty (Group B):    -20 to +18 pts  
      Credibility bonus (Group C):           -5  to +20 pts
      Market signal bonus (Group D):          0  to +11 pts
    
    Final multiplier = base_availability × (1 + (bonuses - penalties) / 100)
    Clamped to 0.25 – 1.35
    """
    sig = candidate["redrob_signals"]

    # ----------------------------------------------------------------
    # GROUP A — AVAILABILITY MULTIPLIER (0.3 – 1.0)
    # ----------------------------------------------------------------
    days_inactive = days_since(sig.get("last_active_date", "2020-01-01"))

    if days_inactive <= 7:
        activity_mult = 1.0
    elif days_inactive <= 30:
        activity_mult = 0.95
    elif days_inactive <= 60:
        activity_mult = 0.90
    elif days_inactive <= 90:
        activity_mult = 0.85
    elif days_inactive <= 180:
        activity_mult = 0.60
    elif days_inactive <= 365:
        activity_mult = 0.30
    else:
        activity_mult = 0.15  # effectively gone

    open_to_work = sig.get("open_to_work_flag", False)
    open_mult    = 1.0 if open_to_work else 0.55

    notice       = sig.get("notice_period_days", 90)
    if notice <= 30:
        notice_mult = 1.0
    elif notice <= 60:
        notice_mult = 0.90
    elif notice <= 90:
        notice_mult = 0.80
    else:
        notice_mult = 0.65

    # Both unverified = contact risk
    verified_email = sig.get("verified_email", True)
    verified_phone = sig.get("verified_phone", True)
    verify_mult    = 0.85 if (not verified_email and not verified_phone) else 1.0

    base_availability = activity_mult * open_mult * notice_mult * verify_mult
    base_availability = max(0.30, min(1.0, base_availability))

    # ----------------------------------------------------------------
    # GROUP B — ENGAGEMENT QUALITY (additive points, -20 to +18)
    # ----------------------------------------------------------------
    engagement_pts = 0

    rr = sig.get("recruiter_response_rate", 0.5)
    if rr < 0.05:
        engagement_pts -= 15
    elif rr < 0.15:
        engagement_pts -= 8
    elif rr > 0.70:
        engagement_pts += 8

    # High response time amplifies low response rate
    art = sig.get("avg_response_time_hours", 48)
    if art > 168 and rr < 0.20:   # >1 week + low response rate
        engagement_pts -= 5

    icr = sig.get("interview_completion_rate", 0.5)
    if icr < 0.50:
        engagement_pts -= 8
    elif icr > 0.85:
        engagement_pts += 5

    oar = sig.get("offer_acceptance_rate", -1)
    if oar == 0.0:
        engagement_pts -= 8   # has offers, declined all
    # oar == -1 means no prior offers → neutral, no change

    apps = sig.get("applications_submitted_30d", 0)
    if apps == 0 and not open_to_work:
        engagement_pts -= 5   # likely unavailable

    engagement_pts = max(-20, min(18, engagement_pts))

    # ----------------------------------------------------------------
    # GROUP C — PROFILE CREDIBILITY (-5 to +20)
    # ----------------------------------------------------------------
    credibility_pts = 0

    github = sig.get("github_activity_score", -1)
    if github > 60:
        credibility_pts += 10
    elif github > 30:
        credibility_pts += 5
    elif github == 0:
        credibility_pts -= 5   # linked but no activity
    # github == -1 → not linked, mild negative
    elif github == -1:
        credibility_pts -= 2

    completeness = sig.get("profile_completeness_score", 80)
    if completeness >= 85:
        credibility_pts += 5
    elif completeness < 50:
        credibility_pts -= 5

    endorsements = sig.get("endorsements_received", 0)
    if endorsements >= 20:
        credibility_pts += 5
    elif endorsements >= 10:
        credibility_pts += 2

    linkedin = sig.get("linkedin_connected", True)
    if not linkedin:
        credibility_pts -= 2   # mildly suspicious for senior professional

    # Salary sanity check — >80 LPA unrealistic for Series A
    salary = sig.get("expected_salary_range_inr_lpa", {})
    if isinstance(salary, dict):
        sal_min = salary.get("min", 0)
        if sal_min > 80:
            credibility_pts -= 5

    credibility_pts = max(-5, min(20, credibility_pts))

    # ----------------------------------------------------------------
    # GROUP D — MARKET DEMAND SIGNALS (0 to +11)
    # ----------------------------------------------------------------
    market_pts = 0

    saved = sig.get("saved_by_recruiters_30d", 0)
    if saved > 5:
        market_pts += 8
    elif saved > 2:
        market_pts += 4

    views = sig.get("profile_views_received_30d", 0)
    if views > 20:
        market_pts += 3

    market_pts = max(0, min(11, market_pts))

    # ----------------------------------------------------------------
    # COMBINE
    # ----------------------------------------------------------------
    total_bonus_pts = engagement_pts + credibility_pts + market_pts
    # Convert pts to a multiplier adjustment: 100 pts = ×1.0 change
    bonus_mult = 1.0 + (total_bonus_pts / 100)

    final_mult = base_availability * bonus_mult
    final_mult = round(max(0.25, min(1.35, final_mult)), 3)

    return final_mult, {
        "days_inactive":    days_inactive,
        "activity_mult":    round(activity_mult, 3),
        "open_mult":        open_mult,
        "notice_mult":      notice_mult,
        "verify_mult":      verify_mult,
        "base_availability": round(base_availability, 3),
        "engagement_pts":   engagement_pts,
        "credibility_pts":  credibility_pts,
        "market_pts":       market_pts,
        "total_bonus_pts":  total_bonus_pts,
        "bonus_mult":       round(bonus_mult, 3),
        "final_mult":       final_mult,
    }


# Quick sanity check on 5 candidates
for c in candidates[:5]:
    mult, detail = score_behavioral(c)
    print(f"{c['candidate_id']}: mult={mult}, base={detail['base_availability']}")


CAND_0000001: mult=0.983, base=0.855
CAND_0000002: mult=0.312, base=0.3
CAND_0000003: mult=0.319, base=0.304
CAND_0000004: mult=0.298, base=0.304
CAND_0000005: mult=0.3, base=0.3


In [9]:
def generate_reasoning(candidate: dict, score_row: dict, jd: dict) -> str:
    profile  = candidate["profile"]
    history  = candidate["career_history"]
    sig      = candidate["redrob_signals"]

    yoe             = profile["years_of_experience"]
    req_hits        = score_row.get("req_hits", [])
    consulting_ratio = score_row.get("consulting_ratio", 0)
    title_reason    = score_row.get("title_reason", "")
    notice          = sig["notice_period_days"]
    response_rate   = sig["recruiter_response_rate"]
    open_to_work    = sig["open_to_work_flag"]
    days_inactive   = score_row.get("days_inactive", 999)

    # ── Sentence 1: Technical signal ─────────────────────────────────
    tools    = [s for s in req_hits if s.lower() in {
        "milvus", "faiss", "pinecone", "weaviate", "elasticsearch",
        "opensearch", "qdrant", "pytorch", "tensorflow", "langchain",
        "lora", "qlora", "peft", "hugging face transformers"
    }]
    concepts = [s for s in req_hits if s not in tools]

    if tools:
        s1 = f"Demonstrated expertise with {', '.join(tools[:3])} ({yoe:.0f} YOE)"
        if concepts:
            s1 += f", focusing on {concepts[0].lower()}"
        s1 += "."
    elif concepts:
        s1 = f"Technical background in {', '.join(concepts[:3])} ({yoe:.0f} YOE)."
    else:
        s1 = f"Adjacent ML/engineering background ({yoe:.0f} YOE)."

    # ── Sentence 2: Production signal ────────────────────────────────
    career_desc = " ".join(h.get("description", "").lower() for h in history)
    has_scale   = any(w in career_desc for w in [
        "scale", "million", "high traffic", "low latency", "qps", "production", "shipped"
    ])
    prod_score = score_row.get("production_sim", 0.0)

    if prod_score >= 0.6 or has_scale:
        s2 = "Proven experience deploying ML systems to production at scale."
    elif prod_score >= 0.35:
        s2 = "Familiar with production ML pipelines and deployment workflows."
    else:
        s2 = "Experience primarily in model development and research."

    # ── Sentence 3: Hiring signal ─────────────────────────────────────
    hiring = []
    if response_rate >= 0.7:
        hiring.append("strong recruiter engagement")
    if sig.get("github_activity_score", -1) > 60:
        hiring.append("active GitHub history")
    if open_to_work and days_inactive <= 30:
        hiring.append("actively looking and recently active")
    elif open_to_work:
        hiring.append("open to work")
    if notice <= 30:
        hiring.append("immediate joiner")
    elif notice <= 60:
        hiring.append(f"{notice}-day notice")

    s3 = ("Candidate shows " + ", ".join(hiring) + "." if hiring
          else "Hiring readiness within standard parameters.")

    # ── Sentence 4: Concerns (optional) ──────────────────────────────
    concerns = []
    if notice > 90:
        concerns.append(f"long notice period ({notice} days)")
    if score_row.get("hype_penalty", 0) > 0.05:
        concerns.append("AI experience skews toward LLM wrapper libraries")
    if consulting_ratio > 0.6:
        concerns.append(f"high consulting exposure ({consulting_ratio:.0%})")
    if title_reason in ("neutral_eng_no_ml_history", "unknown_title_no_ml_history"):
        concerns.append("title not ML-specific")
    if yoe < 3.0:
        concerns.append("below seniority threshold")
    elif yoe >= 15.0:
        concerns.append("experience exceeds target range")

    s4 = ("Note: " + "; ".join(concerns) + "." if concerns else "")

    parts = [s1, s2, s3]
    if s4:
        parts.append(s4)
    return " ".join(parts).replace("..", ".").strip()

In [10]:
def build_candidate_text(candidate: dict) -> str:
    profile = candidate["profile"]
    history = candidate["career_history"]
    skills  = candidate["skills"]
    parts   = []

    if profile.get("headline"):
        parts.append(profile["headline"])

    parts.append(f"{profile['current_title']} at {profile['current_company']}.")

    if profile.get("summary"):
        parts.append(profile["summary"])

    # Skills first — most signal-dense
    skill_parts = []
    for s in skills:
        prof = s.get("proficiency", "")
        dur  = s.get("duration_months", 0)
        if prof in ("advanced", "expert") and dur >= 12:
            skill_parts.append(f"{s['name']} (advanced, {dur} months)")
        else:
            skill_parts.append(s["name"])
    if skill_parts:
        parts.append("Skills: " + ", ".join(skill_parts))

    # Only top 3 most recent roles — MiniLM truncates at ~256 tokens
    for h in history[:3]:
        role_parts = [f"{h['title']} at {h['company']} ({h.get('industry', '')})."]
        if h.get("description"):
            role_parts.append(h["description"])
        parts.append(" ".join(role_parts))

    parts.append(f"Located in {profile.get('location', '')}, {profile.get('country', '')}.")

    # Cap at 1500 chars to stay within model token limit
    text = " ".join(parts)
    return text[:1500]
print("Candidate text builder ready.")


Candidate text builder ready.


In [11]:
from sentence_transformers import SentenceTransformer

# Load model
print("Loading model...")
MODEL_PATH = ".\\models\\BAAI\\bge-small-en-v1.5"
model = SentenceTransformer(MODEL_PATH)
print("Model loaded.")

# Embed JD concepts — specific aspects of the role
RANKING_CONCEPT = """
Built and deployed production ranking or retrieval systems. Dense retrieval,
semantic search, vector databases, FAISS, Pinecone, Weaviate, Qdrant, Milvus,
Elasticsearch, OpenSearch. Recommendation systems, information retrieval,
learning to rank, re-ranking, BM25, hybrid search. Sentence transformers,
bi-encoder models, cross-encoders, embedding models, RAG systems.
"""

EVAL_CONCEPT = """
Designed evaluation frameworks for ranking and retrieval systems. Offline
evaluation metrics NDCG, MRR, MAP, Precision at K. A/B testing for ranking
quality. Online to offline correlation. Statistical significance testing.
Recall evaluation, precision-recall tradeoffs, benchmark datasets.
"""

PRODUCTION_CONCEPT = """
Shipped ML systems to production at scale. Fine-tuning LLMs with LoRA, QLoRA,
PEFT, instruction tuning. Strong Python engineering, production-grade code.
MLOps, experiment tracking, MLflow, Weights and Biases. Model serving,
inference optimization. Open source contributions in AI and ML.
"""

print("Embedding JD concepts...")
concept_embs = {
    "ranking":    model.encode(RANKING_CONCEPT,    normalize_embeddings=True),
    "evaluation": model.encode(EVAL_CONCEPT,       normalize_embeddings=True),
    "production": model.encode(PRODUCTION_CONCEPT, normalize_embeddings=True),
}

# Full JD embedding for broad semantic recall
print("Embedding full JD...")
jd_emb = model.encode(JD_TEXT, normalize_embeddings=True)

print("JD and concept embeddings ready.")


Loading model...
Model loaded.
Embedding JD concepts...
Embedding full JD...
JD and concept embeddings ready.


In [12]:
import os

if os.path.exists("data/candidate_embs.npy") and os.path.exists("data/embedding_id_order.json"):
    print("Cached embeddings found — skipping encode.")
else:
    print(f"Encoding {len(candidates):,} candidates...")
    candidate_texts = [build_candidate_text(c) for c in candidates]
    candidate_embs = model.encode(
        candidate_texts,
        normalize_embeddings=True,
        batch_size=256,
        show_progress_bar=True,
    )
    os.makedirs("data", exist_ok=True)
    np.save("data/candidate_embs.npy", candidate_embs)
    np.save("data/jd_emb.npy", jd_emb)
    np.save("data/ranking_concept.npy",    concept_embs["ranking"])
    np.save("data/eval_concept.npy",       concept_embs["evaluation"])
    np.save("data/production_concept.npy", concept_embs["production"])

    cand_ids = [c["candidate_id"] for c in candidates]
    with open("data/embedding_id_order.json", "w") as f:
        json.dump(cand_ids, f)

    print(f"Done. Shape: {candidate_embs.shape} ({candidate_embs.nbytes/1e6:.0f} MB)")

Cached embeddings found — skipping encode.


In [13]:
%pip install rapidfuzz

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from rapidfuzz import fuzz

RETRIEVAL_KW = {
    "faiss", "pinecone", "weaviate", "qdrant", "milvus", "elasticsearch",
    "opensearch", "vector search", "semantic search", "dense retrieval",
    "rag", "retrieval", "hybrid search", "bm25", "reranking",
    "sentence-transformers", "embedding", "vector database"
}
RANKING_KW = {
    "ranking", "recommendation", "recommender", "learning to rank",
    "ltr", "reranking", "information retrieval", "relevance", "re-ranking"
}
EVAL_KW = {
    "ndcg", "mrr", "map", "a/b test", "offline evaluation",
    "precision@k", "recall@k", "benchmark", "evaluation framework",
    "experiment", "statistical significance"
}
PRODUCTION_KW = {
    "production", "deployed", "shipped", "scale", "latency", "qps",
    "throughput", "mlops", "model serving", "inference", "monitoring",
    "distributed", "real-time", "low latency"
}

LLM_HYPE_KW = {
    "langchain", "llamaindex", "chatgpt", "gpt-4", "openai api",
    "prompt engineering", "chatbot", "llm wrapper", "langflow",
    "gpt-3", "gpt4", "gpt3", "auto-gpt", "agentgpt"
}

def _career_domain_score(career_text: str, keywords: set) -> float:
    hits = sum(1 for kw in keywords if kw in career_text)
    if hits == 0:
        return 0.0
    return min(1.0, hits / 4)


def score_skills_semantic(candidate: dict,
                          candidate_emb: np.ndarray,
                          concept_embs: dict,
                          jd: dict) -> tuple[float, dict]:
    # ── Semantic concept similarities ────────────────────────────────
    ranking_sim    = float(candidate_emb @ concept_embs["ranking"])
    evaluation_sim = float(candidate_emb @ concept_embs["evaluation"])
    production_sim = float(candidate_emb @ concept_embs["production"])

    concept_raw = (
        0.45 * ranking_sim +
        0.30 * evaluation_sim +
        0.25 * production_sim
    )
    concept_score = max(0.0, min(1.0, (concept_raw - 0.15) / 0.45))

    # ── Career keyword evidence ───────────────────────────────────────
    # Scores actual job descriptions — not just skills/summary
    career_text = " ".join(
        h.get("description", "").lower() + " " + h.get("title", "").lower()
        for h in candidate["career_history"]
    )

    retrieval_career = _career_domain_score(career_text, RETRIEVAL_KW)
    eval_career      = _career_domain_score(career_text, EVAL_KW)
    prod_career      = _career_domain_score(career_text, PRODUCTION_KW)

    career_domain_score = (
        0.45 * retrieval_career +
        0.30 * eval_career +
        0.25 * prod_career
    )

    # Blend: semantic is primary, career keywords add grounding
    blended_score = 0.70 * concept_score + 0.30 * career_domain_score

    # ── Explicit skill hits — credibility bonus only (max +0.15) ─────
    skills      = candidate["skills"]
    assessments = candidate["redrob_signals"].get("skill_assessment_scores", {})
    required    = {s.lower() for s in jd["required_skills"]}

    req_hits = []
    for s in skills:
        name = s["name"]
        nl   = name.lower()
        prof = s.get("proficiency", "beginner")
        dur  = s.get("duration_months", 0)
        pm   = {"beginner": 0.4, "intermediate": 0.7,
                "advanced": 1.0, "expert": 1.1}.get(prof, 0.5)
        dm   = min(dur / 36, 1.0)
        w    = pm * (0.5 + 0.5 * dm)
        if name in assessments:
            w += (assessments[name] / 100) * 0.15

        # Exact / substring match
        matched = nl in required or any(nl in r or r in nl for r in required)

        # Fuzzy fallback if no direct match
        if not matched:
            best_ratio = max((fuzz.ratio(nl, r) for r in required), default=0)
            if best_ratio >= 80:
                w = w * (best_ratio / 100.0)  # discount for fuzzy
                matched = True

        if matched:
            req_hits.append((name, round(w, 3)))

    keyword_bonus = min(len(req_hits) * 0.025, 0.15)
    final = min(1.0, blended_score + keyword_bonus)

    return final, {
        "ranking_sim":         round(ranking_sim, 4),
        "evaluation_sim":      round(evaluation_sim, 4),
        "production_sim":      round(production_sim, 4),
        "concept_score":       round(concept_score, 4),
        "career_domain_score": round(career_domain_score, 4),
        "blended_score":       round(blended_score, 4),
        "keyword_bonus":       round(keyword_bonus, 4),
        "req_hits":            [n for n, _ in req_hits],
        "n_req_hits":          len(req_hits),
        "final":               round(final, 4),
    }

print("score_skills_semantic() updated.")

score_skills_semantic() updated.


In [15]:
def _llm_hype_penalty(candidate: dict, sk_score: float) -> float:
    parts = []
    for h in candidate["career_history"]:
        parts.append(h.get("description", "").lower())
        parts.append(h.get("title", "").lower())
    parts.append(candidate["profile"].get("summary", "").lower())
    parts.append(candidate["profile"].get("headline", "").lower())
    parts.append(" ".join(s["name"].lower() for s in candidate["skills"]))
    combined = " ".join(parts)

    hype_hits = sum(1 for kw in LLM_HYPE_KW if kw in combined)
    if hype_hits <= 2:
        return 0.0
    if sk_score >= 0.6:  # real depth present — no penalty
        return 0.0

    scale = (0.6 - sk_score) / 0.6
    return round(min(0.12, scale * 0.12 * min(1.0, hype_hits / 6)), 4)


def _education_bonus(candidate: dict) -> float:
    relevant_fields = {
        "computer science", "machine learning", "artificial intelligence",
        "data science", "mathematics", "statistics", "information technology",
        "computer engineering", "computational"
    }
    for e in candidate.get("education", []):
        field  = e.get("field_of_study", "").lower()
        degree = e.get("degree", "").lower()
        if any(f in field for f in relevant_fields):
            if "phd" in degree or "ph.d" in degree:
                return 0.05
            elif any(m in degree for m in ["master", "m.tech", "m.s", "m.e", "msc"]):
                return 0.03
            elif any(b in degree for b in ["bachelor", "b.tech", "b.e", "b.s", "bsc"]):
                return 0.01
    return 0.0


def get_title_multiplier(candidate: dict, jd: dict) -> tuple[float, str]:
    profile     = candidate["profile"]
    history     = candidate["career_history"]
    good_titles = {t.lower() for t in jd["good_titles"]}
    disq_titles = {t.lower() for t in jd["disqualifier_titles"]}
    neutral_eng = {t.lower() for t in jd.get("neutral_eng_titles", set())}

    current     = profile["current_title"].lower()
    past_titles = [h.get("title", "").lower() for h in history]
    any_past_ml = any(any(gt in pt for gt in good_titles) for pt in past_titles)

    if any(gt in current for gt in good_titles):
        return 1.0, "good_current_title"
    elif any(ne in current for ne in neutral_eng):
        return (0.80, "neutral_eng_with_past_ml") if any_past_ml else (0.50, "neutral_eng_no_ml_history")
    elif any(dt in current for dt in disq_titles):
        return (0.35, "disq_title_but_past_ml") if any_past_ml else (0.10, "disq_title_no_ml_history")
    else:
        return (0.60, "unknown_title_with_past_ml") if any_past_ml else (0.30, "unknown_title_no_ml_history")


def final_score(candidate: dict,
                jd: dict,
                candidate_emb: np.ndarray,
                concept_embs: dict) -> dict:
    """
    Hybrid scoring:
      45% skill/concept score  — semantic concept + career keyword evidence + skill hits
      35% career score         — title, consulting ratio, YoE, stability
      20% JD semantic score    — broad profile-to-JD similarity (safety net)
    Gated by skill score. Title multiplier applied after gate.
    LLM hype penalty and education bonus applied last.
    """
    sk_score, sk_detail = score_skills_semantic(candidate, candidate_emb,
                                                concept_embs, jd)
    ca_score, ca_detail = score_career(candidate, jd)

    # Broad JD similarity — safety net
    jd_sim      = float(candidate_emb @ jd_emb)
    jd_sim_norm = max(0.0, min(1.0, (jd_sim - 0.15) / 0.45))

    base = (
        0.45 * sk_score +
        0.35 * ca_score +
        0.20 * jd_sim_norm
    )

    # Skill gate — weak semantic fit drags everything down
    gate  = sk_score ** 0.5
    gated = base * (0.25 + 0.75 * gate)

    # Education bonus — small, applied before title multiplier
    edu_bonus = _education_bonus(candidate)
    gated = min(1.0, gated + edu_bonus)

    # LLM hype penalty
    hype_pen = _llm_hype_penalty(candidate, sk_score)
    gated = max(0.0, gated - hype_pen)

    title_mult, t_reason = get_title_multiplier(candidate, jd)
    pre_behavioral = round(gated * title_mult, 4)

    return {
        "candidate_id":        candidate["candidate_id"],
        "title":               candidate["profile"]["current_title"],
        "yoe":                 candidate["profile"]["years_of_experience"],
        "location":            candidate["profile"]["location"],
        "country":             candidate["profile"]["country"],
        "skill_score":         round(sk_score, 4),
        "career_score":        round(ca_score, 4),
        "jd_sim":              round(jd_sim_norm, 4),
        "concept_score":       round(sk_detail["concept_score"], 4),
        "career_domain_score": round(sk_detail["career_domain_score"], 4),
        "base_score":          round(base, 4),
        "edu_bonus":           round(edu_bonus, 4),
        "hype_penalty":        round(hype_pen, 4),
        "pre_behavioral":      pre_behavioral,
        "title_mult":          title_mult,
        "title_reason":        t_reason,
        "consulting_ratio":    ca_detail["consulting_ratio"],
        "past_ml_title":       ca_detail["past_ml_title"],
        "req_hits":            sk_detail["req_hits"],
        "n_req_hits":          sk_detail["n_req_hits"],
        "ranking_sim":         sk_detail["ranking_sim"],
        "evaluation_sim":      sk_detail["evaluation_sim"],
    }

print("get_title_multiplier() + final_score() updated.")

get_title_multiplier() + final_score() updated.


In [16]:
def detect_honeypot(candidate: dict) -> tuple[bool, str]:
    profile  = candidate["profile"]
    history  = candidate["career_history"]
    skills   = candidate["skills"]
    sig      = candidate["redrob_signals"]
    yoe      = profile["years_of_experience"]
    flags    = []

    ai_ml = {
        "python","nlp","embeddings","transformers","bert","llm",
        "fine-tuning","lora","qlora","peft","rag","faiss","pinecone",
        "weaviate","qdrant","milvus","elasticsearch","vector search",
        "semantic search","recommendation systems","ranking","bm25",
        "sentence-transformers","hugging face","hugging face transformers",
        "information retrieval","openai","langchain","pytorch","tensorflow",
        "scikit-learn","xgboost","lightgbm","mlflow","weights & biases",
        "kubeflow","feature engineering","deep learning","machine learning",
    }
    csk      = {s["name"].lower() for s in skills}
    ai_count = len(csk & ai_ml)
    comp     = sig.get("profile_completeness_score", 100)
    asmts    = sig.get("skill_assessment_scores", {})

    # FLAG 1 — Many AI skills but very incomplete profile
    if ai_count >= 12 and comp < 50:
        flags.append(f"FLAG1: {ai_count} AI skills, completeness={comp:.0f}")

    # FLAG 2 — Claims core ML but ALL assessments are low
    core_ml = {"python","nlp","embeddings","transformers","fine-tuning",
               "machine learning","deep learning"}
    if bool(csk & core_ml) and len(asmts) > 0 and all(v < 30 for v in asmts.values()):
        flags.append(f"FLAG2: All assessments < 30")

    # FLAG 3 — Non-technical title with many AI skills
    non_tech = {
        "marketing","sales","recruiter","talent acquisition","accountant",
        "finance","hr manager","operations manager","business analyst",
        "content writer","graphic designer","product manager","project manager",
    }
    cur = profile["current_title"].lower()
    if any(nt in cur for nt in non_tech) and ai_count >= 8:
        flags.append(f"FLAG3: Non-tech title + {ai_count} AI skills")

    # FLAG 4 — Claims open to work but zero activity
    otw   = sig.get("open_to_work_flag", False)
    dinac = days_since(sig.get("last_active_date", "2020-01-01"))
    apps  = sig.get("applications_submitted_30d", 0)
    if otw and dinac > 365 and apps == 0:
        flags.append(f"FLAG4: open_to_work=True, inactive {dinac}d, 0 apps")

    # FLAG 5 — All engagement signals impossibly perfect
    rr  = sig.get("recruiter_response_rate", 0)
    art = sig.get("avg_response_time_hours", 999)
    icr = sig.get("interview_completion_rate", 0)
    oar = sig.get("offer_acceptance_rate", -1)
    if rr == 1.0 and art < 0.5 and icr == 1.0 and oar == 1.0:
        flags.append("FLAG5: All engagement signals perfect simultaneously")

    # FLAG 6 — Mentions open source but GitHub is LINKED with zero activity
    # Note: github == -1 means not linked (normal), github == 0 means linked but no activity (suspicious)
    gh  = sig.get("github_activity_score", -1)
    hl  = profile.get("headline", "").lower()
    sm  = profile.get("summary",  "").lower()
    skn = " ".join(s["name"].lower() for s in skills)
    oss = {"open source", "open-source", "github", "contributions"}
    if any(k in hl or k in sm or k in skn for k in oss) and gh == 0:
        flags.append(f"FLAG6: Mentions open source, github linked but score=0")

    # FLAG 7 — Impossible education sequence
    degrees = [
        (e.get("degree","").lower(), e.get("start_year"), e.get("end_year"))
        for e in candidate.get("education", [])
        if e.get("start_year") and e.get("end_year")
    ]
    for i, (d1, s1, e1) in enumerate(degrees):
        for d2, s2, e2 in degrees[i+1:]:
            if ("master" in d2 or "phd" in d2) and ("bachelor" in d1):
                if int(e2) < int(e1):
                    flags.append(f"FLAG7: {d2} ends before {d1}")

    # FLAG 8 — Summary claims retrieval but career has zero evidence
    # Soft signal — kept as a flag but not a hard kill on its own
    summary = candidate["profile"].get("summary", "").lower()
    retrieval_kw = {"retrieval", "search", "rag", "vector search", "milvus", "faiss", "elasticsearch"}
    has_retrieval_summary = any(kw in summary for kw in retrieval_kw)
    career_desc_all = " ".join(
        h.get("description", "").lower() + " " + h.get("title", "").lower()
        for h in candidate["career_history"]
    )
    has_retrieval_career = any(kw in career_desc_all for kw in retrieval_kw)
    if has_retrieval_summary and not has_retrieval_career:
        flags.append("FLAG8: Summary claims retrieval but no career evidence")

    # FLAG 9 — Senior title with impossibly low YoE (threshold tightened to 2.0)
    senior_kw = {"senior", "lead", "staff", "principal", "director", "vp", "head", "architect"}
    if any(sk in cur for sk in senior_kw) and yoe < 2.0:
        flags.append(f"FLAG9: Senior title but YoE={yoe}")

    # FLAG 10 — Graduation vs claimed YoE (only truly impossible gaps)
    edu = candidate.get("education", [])
    grad_years = [e.get("end_year") for e in edu if e.get("end_year")]
    valid_grads = [int(y) for y in grad_years if 1990 <= int(y) <= 2026]
    if valid_grads:
        earliest_grad = min(valid_grads)  # use EARLIEST degree, not latest
        years_since_earliest = REFERENCE_DATE.year - earliest_grad
        # Only flag if they claim more experience than years since their FIRST degree + 5yr buffer
        if yoe > years_since_earliest + 5:
            flags.append(f"FLAG10: Claims {yoe}yr exp but earliest grad {earliest_grad} ({years_since_earliest}yr ago)")
    
    # STRUCTURAL A — Career duration much longer than stated YoE
    total_m   = sum(h.get("duration_months", 0) for h in history)
    total_yrs = total_m / 12
    if total_yrs > yoe + 3:
        flags.append(f"STRUCTURAL_A: career {total_yrs:.1f}yr > stated YoE {yoe}yr")

    # STRUCTURAL B — Stated YoE much higher than career history
    if yoe > total_yrs + 4 and yoe > 8:
        flags.append(f"STRUCTURAL_B: stated YoE {yoe}yr, only {total_yrs:.1f}yr in history")

    # STRUCTURAL C — Current job started before YoE allows
    for h in history:
        if h.get("is_current") and h.get("start_date"):
            try:
                start     = datetime.strptime(h["start_date"], "%Y-%m-%d").date()
                years_ago = (REFERENCE_DATE - start).days / 365
                if years_ago > yoe + 2:
                    flags.append(f"STRUCTURAL_C: current job {years_ago:.1f}yr ago, YoE={yoe}")
            except:
                pass

    # ── Decision logic ───────────────────────────────────────────────
    # In the decision logic section, replace the current is_honeypot line:
    has_flag1  = any("FLAG1"  in f for f in flags)
    has_flag2  = any("FLAG2"  in f for f in flags)
    has_flag3  = any("FLAG3"  in f for f in flags)
    has_flag5  = any("FLAG5"  in f for f in flags)
    has_flag8  = any("FLAG8"  in f for f in flags)
    has_flag10 = any("FLAG10" in f for f in flags)

    has_struct_a = any("STRUCTURAL_A" in f for f in flags)
    has_struct_b = any("STRUCTURAL_B" in f for f in flags)

    # Hard flags only: FLAG8, FLAG10, STRUCTURAL excluded
    hard_flags = [f for f in flags if not any(s in f for s in
                  ["FLAG8", "FLAG10", "STRUCTURAL_A", "STRUCTURAL_B"])]
    n_hard = len(hard_flags)

    # FLAG5 alone is statistically impossible
    instant_kill = has_flag5

    # Strong 2-flag hard combos
    strong_2flag = (has_flag2 and has_flag3)

    # FLAG3 + FLAG8 + FLAG10 together: non-tech title, summary vs career mismatch,
    # AND graduation/YoE mismatch — three independent signals pointing at fraud
    triple_soft_kill = (has_flag3 and has_flag8 and has_flag10)

    struct_b_kill = has_struct_b and n_hard >= 2
    struct_a_kill = has_struct_a and n_hard >= 3

    is_honeypot = (
        instant_kill      or
        n_hard >= 3       or
        strong_2flag      or
        triple_soft_kill  or
        struct_b_kill     or
        struct_a_kill
    )
    reason = f"{len(flags)} flags: " + " | ".join(flags) if flags else ""
    return is_honeypot, reason

print("detect_honeypot() defined.")

detect_honeypot() defined.


In [17]:
import time

def run_pipeline(candidates: list,
                 jd: dict,
                 candidate_embs: np.ndarray,
                 concept_embs: dict,
                 jd_emb: np.ndarray,
                 emb_index: dict,
                 top_n: int = 100,
                 verbose: bool = True) -> pd.DataFrame:

    t0 = time.time()
    def log(msg):
        if verbose:
            print(f"[{time.time()-t0:5.1f}s] {msg}", flush=True)

    # ----------------------------------------------------------------
    # STAGE 1 — Hard disqualifier filter
    # ----------------------------------------------------------------
    log(f"Stage 1: Hard filter — input {len(candidates):,}")
    # Rebuild stage2 fresh exactly as the pipeline does
    disq     = {t.lower() for t in JD["disqualifier_titles"]}
    cf       = {f.lower() for f in JD["consulting_firms"]}
    gt       = {t.lower() for t in JD["good_titles"]}
    required = {s.lower() for s in JD["required_skills"]}

    stage1 = []
    for c in candidates:
        p     = c["profile"]
        title = p["current_title"].lower()
        yoe   = p["years_of_experience"]
        hist  = c["career_history"]
        cos   = [h.get("company", "").lower() for h in hist]
        pts   = [h.get("title",   "").lower() for h in hist]
        if any(d in title for d in disq):
            if not any(any(g in pt for g in gt) for pt in pts):
                continue
        if yoe < JD["yoe_hard_min"]:
            continue
        if cos and len(cos) >= 2 and all(any(f in co for f in cf) for co in cos):
            continue
        total_m   = sum(h.get("duration_months", 0) for h in hist)
        total_yrs = total_m / 12
        if yoe > total_yrs + 4 and yoe > 8:
            continue
        stage1.append(c)

    stage2 = []
    for c in stage1:
        skills = c["skills"]
        has_hit = any(
            s["name"].lower() in required or
            any(s["name"].lower() in r or r in s["name"].lower() for r in required)
            for s in skills
        )
        if has_hit:
            stage2.append(c)

    # Now reproduce stage 3 step by step
    stage2_indices = [emb_index[c["candidate_id"]] for c in stage2]
    stage2_embs    = candidate_embs[stage2_indices]

    raw_sims  = stage2_embs @ jd_emb
    norm_sims = np.clip((raw_sims - 0.10) / 0.45, 0.0, 1.0)

    top_k   = min(600, len(stage2))
    top_idx = np.argsort(norm_sims)[::-1][:top_k]

    # top_idx are positions in stage2_embs / stage2 — print what they resolve to
    print("Top 10 positions in top_idx:", top_idx[:10])
    print("\nWhat stage2[top_idx[i]] resolves to:")
    for i in top_idx[:10]:
        c = stage2[i]
        print(f"  stage2[{i}] = {c['candidate_id']} | sim={norm_sims[i]:.4f} | {c['profile']['current_title']}")

    print(f"\nstage2 length: {len(stage2)}")
    print(f"Max index in top_idx: {top_idx.max()}")
    print(f"Do all top_idx fit within stage2? {top_idx.max() < len(stage2)}")
    # ----------------------------------------------------------------
    # STAGE 3 — Semantic recall using precomputed embeddings
    # ----------------------------------------------------------------
    log("Stage 3: Semantic recall from precomputed embeddings")

    # Get embeddings for stage2 survivors
    stage2_indices = [emb_index[c["candidate_id"]] for c in stage2]
    stage2_embs    = candidate_embs[stage2_indices]

    # Cosine similarity with full JD embedding
    raw_sims   = stage2_embs @ jd_emb
    s_min, s_max = raw_sims.min(), raw_sims.max()
    norm_sims = (raw_sims - s_min) / (s_max - s_min + 1e-8)

    # Keep top 600
    top_k      = min(600, len(stage2))
    top_idx    = np.argsort(norm_sims)[::-1][:top_k]
    stage3     = [(stage2[i], stage2_embs[i], float(norm_sims[i])) for i in top_idx]

    log(f"Stage 3: {len(stage3):,} passed")

    # ----------------------------------------------------------------
    # STAGE 4 — Full hybrid scoring
    # ----------------------------------------------------------------
    log("Stage 4: Full hybrid scoring")

    stage4 = []
    for c, emb, jd_sim_norm in stage3:
        fs = final_score(c, jd, emb, concept_embs)

        # Hard kills
        if fs["title_mult"] <= 0.1:
            continue

        bm, bd = score_behavioral(c)
        rs     = round(fs["pre_behavioral"] * bm, 4)

        stage4.append({
            **fs,
            "beh_multiplier":  bm,
            "days_inactive":   bd["days_inactive"],
            "availability":    bd["base_availability"],
            "engagement_pts":  bd["engagement_pts"],
            "credibility_pts": bd["credibility_pts"],
            "market_pts":      bd["market_pts"],
            "ranked_score":    rs,
        })

    df4 = (pd.DataFrame(stage4)
             .sort_values("ranked_score", ascending=False)
             .reset_index(drop=True))
    log(f"Stage 4: {len(df4):,} scored")

    # ----------------------------------------------------------------
    # STAGE 5 — Honeypot filter on top 300
    # ----------------------------------------------------------------
    log("Stage 5: Honeypot detection")
    lkp   = {c["candidate_id"]: c for c, _, _ in stage3}
    clean = []
    hp_n  = 0
    for _, row in df4.head(300).iterrows():
        is_hp, _ = detect_honeypot(lkp[row["candidate_id"]])
        if is_hp:
            hp_n += 1
        else:
            clean.append(row["candidate_id"])

    hp_rate = hp_n / min(300, len(df4))
    log(f"Stage 5: {hp_n} honeypots removed (rate={hp_rate:.1%})")
    if hp_rate > 0.10:
        log("WARNING: Honeypot rate > 10% — disqualification risk!")

    df5 = df4[df4["candidate_id"].isin(clean)].reset_index(drop=True)

    # ----------------------------------------------------------------
    # STAGE 6 — Top 100 + reasoning
    # ----------------------------------------------------------------
    log("Stage 6: Generating reasoning")
    df6         = df5.head(top_n).copy()
    df6["rank"] = range(1, len(df6) + 1)
    df6["reasoning"] = [
        generate_reasoning(lkp[row["candidate_id"]], row.to_dict(), jd)
        for _, row in df6.iterrows()
    ]

    log(f"Done — {len(df6)} candidates in final output")
    return df6

print("run_pipeline() defined.")

run_pipeline() defined.


In [18]:
# Load precomputed embeddings from disk
print("Loading embeddings...")
candidate_embs = np.load("data/candidate_embs.npy")
jd_emb         = np.load("data/jd_emb.npy")
concept_embs   = {
    "ranking":    np.load("data/ranking_concept.npy"),
    "evaluation": np.load("data/eval_concept.npy"),
    "production": np.load("data/production_concept.npy"),
}
with open("data/embedding_id_order.json") as f:
    emb_id_order = json.load(f)

emb_index = {cid: i for i, cid in enumerate(emb_id_order)}
print(f"Embeddings loaded: {candidate_embs.shape}")

# Run pipeline
print("\nRunning pipeline...\n")
df_final = run_pipeline(
    candidates     = candidates,
    jd             = JD,
    candidate_embs = candidate_embs,
    concept_embs   = concept_embs,
    jd_emb         = jd_emb,
    emb_index      = emb_index,
    top_n          = 100,
    verbose        = True,
)

assert isinstance(df_final["req_hits"].iloc[0], list), "req_hits lost its type"

# Show results
cols = ["rank", "candidate_id", "title", "yoe",
        "skill_score", "career_score", "ranking_sim",
        "title_mult", "beh_multiplier", "ranked_score"]

print(f"\nTotal output: {len(df_final)} candidates")
print("\n=== Top 15 ===")
print(df_final.head(15)[cols].to_string(index=False))

# Save submission
out = df_final[["candidate_id", "rank", "ranked_score", "reasoning"]].copy()
out.columns = ["candidate_id", "rank", "score", "reasoning"]
out.to_csv("output/submission_v5.csv", index=False, encoding="utf-8")
print("Saved with correct column order: candidate_id, rank, score, reasoning")

Loading embeddings...
Embeddings loaded: (100000, 384)

Running pipeline...

[  0.0s] Stage 1: Hard filter — input 100,000
Top 10 positions in top_idx: [14728 14727 14726 14725 14724 14723 14722 14721 14720 14719]

What stage2[top_idx[i]] resolves to:
  stage2[14728] = CAND_0099999 | sim=1.0000 | .NET Developer
  stage2[14727] = CAND_0099998 | sim=1.0000 | Analytics Engineer
  stage2[14726] = CAND_0099994 | sim=1.0000 | Full Stack Developer
  stage2[14725] = CAND_0099993 | sim=1.0000 | Cloud Engineer
  stage2[14724] = CAND_0099991 | sim=1.0000 | DevOps Engineer
  stage2[14723] = CAND_0099984 | sim=1.0000 | Analytics Engineer
  stage2[14722] = CAND_0099983 | sim=1.0000 | Data Engineer
  stage2[14721] = CAND_0099982 | sim=1.0000 | Full Stack Developer
  stage2[14720] = CAND_0099981 | sim=1.0000 | Full Stack Developer
  stage2[14719] = CAND_0099977 | sim=1.0000 | Senior Software Engineer

stage2 length: 14729
Max index in top_idx: 14728
Do all top_idx fit within stage2? True
[  3.6s] Stag

## Validation

### Honeypot

In [19]:
hp_ids = set()
for c in candidates:
    is_hp, _ = detect_honeypot(c)
    if is_hp:
        hp_ids.add(c["candidate_id"])

print(f"Total honeypots detected: {len(hp_ids)}")

Total honeypots detected: 78


In [20]:
hp_records = []
for c in candidates:
    is_hp, reason = detect_honeypot(c)
    if is_hp:
        profile = c["profile"]
        sig     = c["redrob_signals"]
        salary  = sig.get("expected_salary_range_inr_lpa", {})

        hp_records.append({
            # --- Identity ---
            "candidate_id":                  c["candidate_id"],
            "reason":                        reason,

            # --- Profile ---
            "current_title":                 profile.get("current_title"),
            "yoe":                           profile.get("years_of_experience"),
            "location":                      profile.get("location"),
            "country":                       profile.get("country"),
            "headline":                      profile.get("headline"),
            "summary":                       profile.get("summary"),

            # --- Skills ---
            "skills":                        ", ".join(s["name"] for s in c.get("skills", [])),

            # --- Career history ---
            "career_history":                " | ".join(
                f"{h.get('title')} @ {h.get('company')} ({h.get('duration_months')}mo)"
                for h in c.get("career_history", [])
            ),
            "career_descriptions":           " | ".join(
                h.get("description", "") for h in c.get("career_history", [])
            ),

            # --- Education ---
            "education":                     " | ".join(
                f"{e.get('degree')} @ {e.get('institution')} ({e.get('start_year')}-{e.get('end_year')})"
                for e in c.get("education", [])
            ),

            # --- RedrRob signals ---
            "last_active_date":              sig.get("last_active_date"),
            "open_to_work":                  sig.get("open_to_work_flag"),
            "notice_period_days":            sig.get("notice_period_days"),
            "verified_email":                sig.get("verified_email"),
            "verified_phone":                sig.get("verified_phone"),
            "recruiter_response_rate":       sig.get("recruiter_response_rate"),
            "avg_response_time_hours":       sig.get("avg_response_time_hours"),
            "interview_completion_rate":     sig.get("interview_completion_rate"),
            "offer_acceptance_rate":         sig.get("offer_acceptance_rate"),
            "applications_submitted_30d":    sig.get("applications_submitted_30d"),
            "github_activity_score":         sig.get("github_activity_score"),
            "profile_completeness_score":    sig.get("profile_completeness_score"),
            "endorsements_received":         sig.get("endorsements_received"),
            "linkedin_connected":            sig.get("linkedin_connected"),
            "saved_by_recruiters_30d":       sig.get("saved_by_recruiters_30d"),
            "profile_views_received_30d":    sig.get("profile_views_received_30d"),
            "skill_assessment_scores":       json.dumps(sig.get("skill_assessment_scores", {})),
            "salary_min_lpa":                salary.get("min") if isinstance(salary, dict) else None,
            "salary_max_lpa":                salary.get("max") if isinstance(salary, dict) else None,
        })

df_honeypots = pd.DataFrame(hp_records)
df_honeypots.to_csv("output/honeypots.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(df_honeypots)} honeypots to output/honeypots.csv")

Saved 78 honeypots to output/honeypots.csv


In [21]:
ids_in_pool = [int(c["candidate_id"].split("_")[1]) for c in candidates]
print(f"Pool ID range: {min(ids_in_pool)} – {max(ids_in_pool)}")

ids_in_submission = [int(cid.split("_")[1]) for cid in df_final["candidate_id"]]
print(f"Submission ID range: {min(ids_in_submission)} – {max(ids_in_submission)}")
print(f"All from last 5k: {all(i >= 95000 for i in ids_in_submission)}")

Pool ID range: 1 – 100000
Submission ID range: 31 – 99806
All from last 5k: False


In [22]:
print(f"Total candidates loaded: {len(candidates)}")
print(f"Total in emb_index: {len(emb_index)}")
print(f"candidate_embs shape: {candidate_embs.shape}")

# Check what IDs are in the index
emb_ids = list(emb_index.keys())
emb_id_nums = [int(i.split("_")[1]) for i in emb_ids]
print(f"\nID range in emb_index: {min(emb_id_nums)} – {max(emb_id_nums)}")
print(f"IDs below 50000 in emb_index: {sum(1 for i in emb_id_nums if i < 50000)}")
print(f"IDs above 95000 in emb_index: {sum(1 for i in emb_id_nums if i > 95000)}")

# Check how many candidates can actually be looked up
matched = sum(1 for c in candidates if c["candidate_id"] in emb_index)
print(f"\nCandidates with an emb_index entry: {matched} / {len(candidates)}")

Total candidates loaded: 100000
Total in emb_index: 100000
candidate_embs shape: (100000, 384)

ID range in emb_index: 1 – 100000
IDs below 50000 in emb_index: 49999
IDs above 95000 in emb_index: 5000

Candidates with an emb_index entry: 100000 / 100000


In [23]:
# Trace exactly how many candidates survive each stage manually
disq = {t.lower() for t in JD["disqualifier_titles"]}
cf   = {f.lower() for f in JD["consulting_firms"]}
gt   = {t.lower() for t in JD["good_titles"]}
required = {s.lower() for s in JD["required_skills"]}

stage1 = []
stage1_killed = {"disq_title": 0, "yoe": 0, "all_consulting": 0, "structural_b": 0}

for c in candidates:
    p     = c["profile"]
    title = p["current_title"].lower()
    yoe   = p["years_of_experience"]
    hist  = c["career_history"]
    cos   = [h.get("company", "").lower() for h in hist]
    pts   = [h.get("title",   "").lower() for h in hist]

    if any(d in title for d in disq):
        if not any(any(g in pt for g in gt) for pt in pts):
            stage1_killed["disq_title"] += 1
            continue
    if yoe < JD["yoe_hard_min"]:
        stage1_killed["yoe"] += 1
        continue
    if cos and len(cos) >= 2 and all(any(f in co for f in cf) for co in cos):
        stage1_killed["all_consulting"] += 1
        continue
    total_m   = sum(h.get("duration_months", 0) for h in hist)
    total_yrs = total_m / 12
    if yoe > total_yrs + 4 and yoe > 8:
        stage1_killed["structural_b"] += 1
        continue
    stage1.append(c)

print(f"Stage 1 input:  {len(candidates):,}")
print(f"Stage 1 output: {len(stage1):,}")
print(f"Killed by disq_title:    {stage1_killed['disq_title']:,}")
print(f"Killed by yoe:           {stage1_killed['yoe']:,}")
print(f"Killed by all_consulting:{stage1_killed['all_consulting']:,}")
print(f"Killed by structural_b:  {stage1_killed['structural_b']:,}")

stage2 = []
for c in stage1:
    skills = c["skills"]
    has_hit = any(
        s["name"].lower() in required or
        any(s["name"].lower() in r or r in s["name"].lower() for r in required)
        for s in skills
    )
    if has_hit:
        stage2.append(c)

print(f"\nStage 2 input:  {len(stage1):,}")
print(f"Stage 2 output: {len(stage2):,}")
print(f"Killed by skill filter: {len(stage1) - len(stage2):,}")

# Check ID distribution after each stage
s1_ids = [int(c["candidate_id"].split("_")[1]) for c in stage1]
s2_ids = [int(c["candidate_id"].split("_")[1]) for c in stage2]
print(f"\nStage 1 ID range: {min(s1_ids)} – {max(s1_ids)}")
print(f"Stage 1 IDs > 95000: {sum(1 for i in s1_ids if i > 95000)}")
print(f"\nStage 2 ID range: {min(s2_ids)} – {max(s2_ids)}")
print(f"Stage 2 IDs > 95000: {sum(1 for i in s2_ids if i > 95000)}")

Stage 1 input:  100,000
Stage 1 output: 27,491
Killed by disq_title:    63,108
Killed by yoe:           7,162
Killed by all_consulting:2,226
Killed by structural_b:  13

Stage 2 input:  27,491
Stage 2 output: 14,729
Killed by skill filter: 12,762

Stage 1 ID range: 1 – 99999
Stage 1 IDs > 95000: 1321

Stage 2 ID range: 1 – 99999
Stage 2 IDs > 95000: 707


In [24]:
# Reproduce Stage 3 exactly and check ID distribution of top 600
stage2_indices = [emb_index[c["candidate_id"]] for c in stage2]
stage2_embs    = candidate_embs[stage2_indices]

raw_sims   = stage2_embs @ jd_emb
norm_sims  = np.clip((raw_sims - 0.10) / 0.45, 0.0, 1.0)

top_k   = min(600, len(stage2))
top_idx = np.argsort(norm_sims)[::-1][:top_k]
stage3  = [stage2[i] for i in top_idx]

s3_ids = [int(c["candidate_id"].split("_")[1]) for c in stage3]
print(f"Stage 3 output: {len(stage3)}")
print(f"ID range: {min(s3_ids)} – {max(s3_ids)}")
print(f"IDs > 95000: {sum(1 for i in s3_ids if i > 95000)}")
print(f"IDs < 50000: {sum(1 for i in s3_ids if i < 50000)}")

# Check raw similarity stats
print(f"\nRaw sim stats across stage2:")
print(f"  min:  {raw_sims.min():.4f}")
print(f"  max:  {raw_sims.max():.4f}")
print(f"  mean: {raw_sims.mean():.4f}")
print(f"  std:  {raw_sims.std():.4f}")

# Check top 10 by similarity
top10_idx = np.argsort(raw_sims)[::-1][:10]
print(f"\nTop 10 by raw cosine sim:")
for i in top10_idx:
    c = stage2[i]
    print(f"  {c['candidate_id']} | sim={raw_sims[i]:.4f} | {c['profile']['current_title']}")

# Check jd_emb shape and norm
print(f"\njd_emb shape: {jd_emb.shape}")
print(f"jd_emb norm:  {np.linalg.norm(jd_emb):.4f}")
print(f"candidate_embs norm sample (first 5): {[round(np.linalg.norm(candidate_embs[i]), 4) for i in range(5)]}")

Stage 3 output: 600
ID range: 95828 – 99999
IDs > 95000: 600
IDs < 50000: 0

Raw sim stats across stage2:
  min:  0.6530
  max:  0.8660
  mean: 0.7375
  std:  0.0307

Top 10 by raw cosine sim:
  CAND_0079064 | sim=0.8660 | Senior Data Scientist
  CAND_0002025 | sim=0.8607 | Senior AI Engineer
  CAND_0046064 | sim=0.8591 | Senior NLP Engineer
  CAND_0055905 | sim=0.8588 | Senior Machine Learning Engineer
  CAND_0011687 | sim=0.8560 | Senior NLP Engineer
  CAND_0049896 | sim=0.8550 | Search Engineer
  CAND_0008425 | sim=0.8548 | Senior NLP Engineer
  CAND_0088025 | sim=0.8522 | Staff Machine Learning Engineer
  CAND_0048265 | sim=0.8518 | AI Research Engineer
  CAND_0094759 | sim=0.8514 | Lead AI Engineer

jd_emb shape: (384,)
jd_emb norm:  1.0000
candidate_embs norm sample (first 5): [np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(1.0), np.float32(1.0)]


In [25]:
# Check what positions emb_index is actually pointing to for a sample of candidates
sample_candidates = [c["candidate_id"] for c in stage2[:20]]
sample_indices    = [emb_index[cid] for cid in sample_candidates]

print("candidate_id → emb_index position:")
for cid, idx in zip(sample_candidates, sample_indices):
    id_num = int(cid.split("_")[1])
    print(f"  {cid} (ID={id_num:6d}) → index {idx}")

# The smoking gun: are index positions correlated with candidate ID number?
all_stage2_indices = [emb_index[c["candidate_id"]] for c in stage2]
all_stage2_idnums  = [int(c["candidate_id"].split("_")[1]) for c in stage2]

import numpy as np
correlation = np.corrcoef(all_stage2_idnums, all_stage2_indices)[0, 1]
print(f"\nCorrelation between candidate ID number and emb_index position: {correlation:.4f}")
print("(If close to 1.0, the index is just the ID number — off by one or similar)")

# Check what emb_index actually contains for a known good candidate
print(f"\nemb_index['CAND_0002025'] = {emb_index.get('CAND_0002025', 'NOT FOUND')}")
print(f"emb_index['CAND_0099999'] = {emb_index.get('CAND_0099999', 'NOT FOUND')}")

candidate_id → emb_index position:
  CAND_0000001 (ID=     1) → index 0
  CAND_0000010 (ID=    10) → index 9
  CAND_0000014 (ID=    14) → index 13
  CAND_0000015 (ID=    15) → index 14
  CAND_0000031 (ID=    31) → index 30
  CAND_0000032 (ID=    32) → index 31
  CAND_0000035 (ID=    35) → index 34
  CAND_0000038 (ID=    38) → index 37
  CAND_0000043 (ID=    43) → index 42
  CAND_0000044 (ID=    44) → index 43
  CAND_0000060 (ID=    60) → index 59
  CAND_0000062 (ID=    62) → index 61
  CAND_0000063 (ID=    63) → index 62
  CAND_0000082 (ID=    82) → index 81
  CAND_0000088 (ID=    88) → index 87
  CAND_0000091 (ID=    91) → index 90
  CAND_0000096 (ID=    96) → index 95
  CAND_0000110 (ID=   110) → index 109
  CAND_0000112 (ID=   112) → index 111
  CAND_0000116 (ID=   116) → index 115

Correlation between candidate ID number and emb_index position: 1.0000
(If close to 1.0, the index is just the ID number — off by one or similar)

emb_index['CAND_0002025'] = 2024
emb_index['CAND_0099999

In [26]:
# Compare embedding norms and variance across the array
# Valid embeddings should have consistent non-trivial values
# Stale/zero rows will look flat

import numpy as np

# Sample norms from different regions
regions = [
    ("rows 0–999",       candidate_embs[0:1000]),
    ("rows 10000–10999", candidate_embs[10000:11000]),
    ("rows 50000–50999", candidate_embs[50000:51000]),
    ("rows 90000–90999", candidate_embs[90000:91000]),
    ("rows 95000–95999", candidate_embs[95000:96000]),
    ("rows 99000–99999", candidate_embs[99000:100000]),
]

print("Region                  | mean norm | std of rows | mean abs value")
for label, chunk in regions:
    norms    = np.linalg.norm(chunk, axis=1)
    std_rows = chunk.std(axis=0).mean()
    mean_abs = np.abs(chunk).mean()
    print(f"{label} | {norms.mean():.4f}     | {std_rows:.6f}    | {mean_abs:.6f}")

# Also check dot product with jd_emb directly per region
print("\nMean cosine sim with jd_emb per region:")
for label, chunk in regions:
    sims = chunk @ jd_emb
    print(f"  {label}: mean={sims.mean():.4f}, max={sims.max():.4f}")

Region                  | mean norm | std of rows | mean abs value
rows 0–999 | 1.0000     | 0.027445    | 0.034757
rows 10000–10999 | 1.0000     | 0.027381    | 0.034738
rows 50000–50999 | 1.0000     | 0.027378    | 0.034708
rows 90000–90999 | 1.0000     | 0.027397    | 0.034753
rows 95000–95999 | 1.0000     | 0.027446    | 0.034760
rows 99000–99999 | 1.0000     | 0.027288    | 0.034690

Mean cosine sim with jd_emb per region:
  rows 0–999: mean=0.7221, max=0.8409
  rows 10000–10999: mean=0.7235, max=0.8262
  rows 50000–50999: mean=0.7223, max=0.8264
  rows 90000–90999: mean=0.7209, max=0.8343
  rows 95000–95999: mean=0.7227, max=0.8295
  rows 99000–99999: mean=0.7233, max=0.8271
